# TP与SP

TP 拆分的层（如 Linear）需要 All-Gather 输出，但 LayerNorm/Dropout 不需要，所以可以在序列维度分片

## 核心概念：TP 拆分的两种基本方式
在 Megatron-LM 等框架中，TP 主要沿着 张量的最后一个维度（通常是 hidden dimension）进行拆分。但这里讨论的是一种更灵活的优化：在 序列维度（sequence dimension, dim=0 或 dim=1） 上进行分片。

### 为什么要在序列维度分片？
Transformer 的计算特点是：
- Attention：<font color='red'>需要在序列维度上做全局交互（Softmax 归一化涉及整个序列）</font>
- Feed-Forward (FFN)：<font color='red'>每个 token 独立计算，不需要跨 token 的信息</font>

因此，FFN 中的某些操作可以在序列维度分片，而 Attention 部分不行。

## 关键区分：哪些层需要 All-Gather？
### 🔴 需要 All-Gather 的层：Linear / MatMul
原因：线性层的权重矩阵 W  的 shape 是 [input_dim, output_dim]。如果输入 X  在序列维度被分片，每个 TP rank 只拿到部分 token：Y=X⋅W 
- 每个 rank 计算：Y_local=X_local⋅W 
- 结果 Y_local的 shape 是 [local_seq_len, output_dim]
- 但 Y  的每个元素都依赖于完整的 W ，而 W  已经在列维度被拆分了

等等，这里需要更精确的理解。让我重新梳理：

#### <font color='red'>例子 ：Linear 层为什么不能直接在序列维度分片</font>

假设：我们要做 Y=X⋅W ，其中 W:[4,8] （输入4维，输出8维）

##### 标准 Column Parallel：
- <font color='red'>W  按列拆分</font>：
  - Rank 0 拿 W[:,<font color='red'>0:4</font>] ，
  - Rank 1 拿 W[:,<font color='red'>4:8</font>] 
- 输入 X  需要 完整复制 到两个 rank

In [ ]:
# Rank 0
Y0 = X @ W[:, 0:4]   # [2, 8, 4]

# Rank 1  
Y1 = X @ W[:, 4:8]   # [2, 8, 4]

# 结果 Y = [Y0 | Y1] 在列维度拼接，shape: [2, 8, 8]

##### 问题：如果 X  已经在序列维度分片了怎么办？

In [ ]:
Rank 0: X0 = X[:, 0:4, :]   # [2, 4, 4]  <- 只有前半段序列
Rank 1: X1 = X[:, 4:8, :]   # [2, 4, 4]  <- 只有后半段序列

##### 尝试直接计算：

In [ ]:
# Rank 0
Y0 = X0 @ W[:, 0:4]   # [2, 4, 4]  <- 只有前半段序列的部分输出特征

# Rank 1
Y1 = X1 @ W[:, 4:8]   # [2, 4, 4]  <- 只有后半段序列的部分输出特征

##### 问题：
- Rank 0 没有 "学"、"习" 的 token，无法计算它们的输出
- Rank 0 只有输出特征的前4维，后4维在 Rank 1
#### 解决方案：All-Gather

In [ ]:
# Step 1: All-Gather 输入，恢复完整序列
X_full = all_gather([X0, X1], dim=1)   # [2, 8, 4] 每个 rank 都有

# Step 2: 正常做 Column Parallel Linear
Y0 = X_full @ W[:, 0:4]   # [2, 8, 4]
Y1 = X_full @ W[:, 4:8]   # [2, 8, 4]

#### <font color='red'>Attention 中的序列维度问题 例子 6：Self-Attention 的 QK^T</font>
- Attention 公式：
Attention(Q,K,V)=softmax($\frac{QK^T}{\sqrt{d_k}}$ )V 
- Q, K, V 的形状：[batch, seq_len, hidden_dim]
- QK^T 的形状：[batch, seq_len, seq_len]

In [ ]:
        K^T
         ↓
    我  爱  深  度  学  习
Q → 我 [a,  b,  c,  d,  e,  f]   <- "我" 与所有token的注意力
  → 爱 [g,  h,  i,  j,  k,  l]   <- "爱" 与所有token的注意力
  → 深 [m,  n,  o,  p,  q,  r]   <- "深" 与所有token的注意力
  → ...

##### 如果序列维度分片：

In [ ]:
Rank 0 有 Q0: [我, 爱, 深, 度]    (前4个token的Query)
Rank 1 有 Q1: [学, 习, pad, pad]  (后4个token的Query)

Rank 0 有 K0: [我, 爱, 深, 度]    (前4个token的Key)
Rank 1 有 K1: [学, 习, pad, pad]  (后4个token的Key)

##### 计算 QK^T 的问题：
"我" 与 "学" 的注意力分数 = dot(Q[我], K[学])
- Q[我] 在 Rank 0
- K[学] 在 Rank 1
##### <font color='red'>必须通信才能计算！</font>
这就是 Ring Attention 或 All-Gather 在 Attention 中的必要性。



#### 正确的理解方式
在标准的 Megatron TP 中：
- Column Parallel Linear：<font color='red'>权重 W  按列拆分，<b>输入 X  完整复制</b>到所有 ranks</font>
  - 每个 rank 计算 Y_i=X⋅W_i
  - 输出 Y=[Y_1∣Y 2∣...∣Y n]  在列维度分片
  - 不需要 All-Gather，输出天然分片
  
- Row Parallel Linear：<font color='red'>权重 W  按行拆分，输入 X  在列维度分片</font>
  - 每个 rank 计算 Yi=Xi⋅Wi
  - 需要 All-Reduce 来聚合结果
  
但这里讨论的是另一种情况：输入在序列维度分片。

### 🟢 不需要 All-Gather 的层：LayerNorm / Dropout / Activation
这些层的特点是：<font color='blue'>每个 token 独立计算，不涉及跨 token 的操作。</font>
1. LayerNorm
LayerNorm(x)=$γ⋅\frac{x-μ}{\sqrt{σ^2 + ϵ}} + β $
- 统计量 μ 和 σ^2是在 feature dimension 上计算的（对每个 token 单独做）
- 如果输入在序列维度分片，每个 rank 上的 token 可以独立计算自己的 LayerNorm
- <font color='red'>不需要任何通信</font>
2. Dropout
Dropout(x)=x⋅mask 
- 完全是 element-wise 操作
- 每个 token 独立应用 mask
- <font color='red'>不需要通信</font>
3. Activation Functions (GELU, ReLU 等)
GELU(x)=x⋅Φ(x) 
- element-wise 操作
- <font color='red'>不需要通信</font>


#### <font color='red'>例子 LayerNorm 在序列维度分片后的计算</font>
LayerNorm(x)=$γ⋅\frac{x-μ}{\sqrt{σ^2 + ϵ}} + β $
- 关键：μ  和 σ  是在 特征维度（dim=-1）上计算的！

In [ ]:
# Rank 0 上的计算
# 输入: [2, 4, 4]
# 对 "我" 这个token: x = [0.2, 0.5, -0.1, 0.8]
mu = mean([0.2, 0.5, -0.1, 0.8]) = 0.35
sigma = std([0.2, 0.5, -0.1, 0.8]) = 0.38
output = gamma * (x - 0.35) / (0.38 + eps) + beta

# 对 "爱" 这个token: x = [0.1, 0.9, 0.3, 0.2]
mu = mean([0.1, 0.9, 0.3, 0.2]) = 0.375
sigma = std([0.1, 0.9, 0.3, 0.2]) = 0.33
output = gamma * (x - 0.375) / (0.33 + eps) + beta

# ... 每个token独立计算

##### 为什么不需要通信？
- <font color='red'>因为 "我" 的 LayerNorm 只依赖于 "我" 自己的4个特征值，不需要知道 "爱" 或 "学" 的值。</font>
- <font color='red'>即使 "学" 在 Rank 1 上，也完全不影响 "我" 的计算！</font>

## 具体例子：一个 Transformer Block 的 TP 拆分
### 标准 Transformer Block 结构

In [ ]:
Input (seq_len, hidden_dim)
    │
    ├──► LayerNorm ──► Attention ──► Dropout ──┐
    │                                          │
    └──────────────────────────────────────────┘ (残差连接)
                              │
    ├──► LayerNorm ──► Linear ──► GELU ──► Linear ──► Dropout ──┐
    │                                                             │
    └─────────────────────────────────────────────────────────────┘ (残差连接)

### 优化后的分片策略
假设我们有 2 个 TP ranks，输入在序列维度分片：

In [ ]:
Rank 0:  tokens [0 : seq_len/2]    (前半段序列)
Rank 1:  tokens [seq_len/2 : seq_len]  (后半段序列)

####  第一层：Pre-Attention LayerNorm

In [ ]:
Input: [seq_len/2, hidden_dim] on each rank
    │
    ▼
LayerNorm: 每个 token 独立计算
    │
    ▼
Output: [seq_len/2, hidden_dim] on each rank

✅ 不需要 All-Gather — 每个 rank 独立计算自己的 token

#### 第二层：Attention（QKV Linear + Attention）

In [ ]:
Input: [seq_len/2, hidden_dim] on each rank
    │
    ├──► QKV Linear (Column Parallel)
    │       - 需要 All-Gather 输入，因为 Q/K/V 的权重是完整复制的
    │       - 或者：如果 QKV 权重也按列拆分，输入需要广播
    │
    ▼
Attention: 需要完整的 Q, K, V 矩阵
    - 这里问题来了：Attention 的 Softmax 需要全局信息
    - 序列维度分片后，Softmax 的分母需要跨 rank 聚合

🔴 Attention 部分复杂 — 需要 Ring Attention 或其他技术来处理序列并行

#### 第三层：Post-Attention Dropout

In [ ]:
Input: [seq_len/2, hidden_dim] on each rank
    │
    ▼
Dropout: element-wise
    │
    ▼
Output: [seq_len/2, hidden_dim] on each rank

✅ 不需要 All-Gather

#### 第四层：Pre-FFN LayerNorm

In [ ]:
Input: [seq_len/2, hidden_dim] on each rank
    │
    ▼
LayerNorm: 每个 token 独立
    │
    ▼
Output: [seq_len/2, hidden_dim] on each rank

✅ 不需要 All-Gather

#### 第五层：FFN 第一个 Linear (Column Parallel)

In [ ]:
Input: [seq_len/2, hidden_dim] on each rank  (完整复制到所有 ranks)
    │
    ├──► Linear: Y = X · W
    │       W 的 shape: [hidden_dim, 4*hidden_dim/n]
    │       每个 rank 计算部分输出特征
    │
    ▼
Output: [seq_len/2, 4*hidden_dim/n] on each rank

🔴 需要 All-Gather — 如果下一个操作需要完整的特征维度
等等，这里标准 Megatron 的做法是：
- Column Parallel Linear 的输出天然在列维度分片
- 下一个 Row Parallel Linear 的输入接受分片输入
- 所以中间不需要 All-Gather
让我重新理解原问题：

## 重新理解：原问题的真正含义
原问题说的是：
- "TP 拆分的层（如 Linear）需要 All-Gather 输出，但 LayerNorm/Dropout 不需要，所以可以在序列维度分片"
这里的场景应该是：
### 场景：在序列维度做 Sequence Parallelism (SP)
在 DeepSpeed Ulysses 或 Megatron-LM 的 Sequence Parallelism 中：
1. 输入按序列维度分片给各个 TP ranks
2. Attention 层：<font color='red'>需要特殊的 Ring Attention 来处理跨 rank 的注意力计算</font>
3. FFN 层：
   - LayerNorm：
     - 可以在序列维度分片，每个 token 独立
   - Linear：
     - 输入需要在进入 Linear 前 All-Gather（因为 Linear 的权重是按 hidden_dim 拆分的，需要完整的 hidden_dim 维度）
   - Dropout/GELU：
     - 可以在序列维度分片

## 详细例子：Sequence Parallelism 中的 FFN
### 设置
- Batch size: 2
- Sequence length: 4096
- Hidden dim: 4096
- TP size: 2
- 输入 X  在序列维度分片
### 输入分片

In [ ]:
原始输入 X: [2, 4096, 4096]

Rank 0: X0 = X[:, 0:2048, :]    shape: [2, 2048, 4096]
Rank 1: X1 = X[:, 2048:4096, :] shape: [2, 2048, 4096]

### Step 1: LayerNorm（序列维度分片）

In [ ]:
# Rank 0
mu_0 = mean(X0, dim=-1, keepdim=True)      # [2, 2048, 1]
sigma_0 = std(X0, dim=-1, keepdim=True)   # [2, 2048, 1]
Y0 = gamma * (X0 - mu_0) / (sigma_0 + eps) + beta

# Rank 1
mu_1 = mean(X1, dim=-1, keepdim=True)      # [2, 2048, 1]
sigma_1 = std(X1, dim=-1, keepdim=True)   # [2, 2048, 1]
Y1 = gamma * (X1 - mu_1) / (sigma_1 + eps) + beta

✅ 不需要通信 — 每个 token 的统计量只依赖于自己的特征

### Step 2: 第一个 Linear（需要 All-Gather）
Linear 的权重 W1 shape: [4096, 16384] (4x expansion)

在标准 TP 中，W1按列拆分：
- Rank 0: W_(1,0) shape: [4096, 8192]
- Rank 1: W_(1,1) shape: [4096, 8192]

#### 问题：
  - 输入 Y  在序列维度分片，但 Linear 需要完整的 hidden_dim 维度（4096），且输出要在列维度分片。
#### 解决方案：

In [ ]:
# 先 All-Gather 序列维度的输入
Y_full = all_gather(Y, dim=1)   # [2, 4096, 4096] 现在每个 rank 都有完整序列

# 然后做 Column Parallel Linear
Z0 = Y_full @ W_{1,0}   # [2, 4096, 8192] on Rank 0
Z1 = Y_full @ W_{1,1}   # [2, 4096, 8192] on Rank 1

🔴 需要 All-Gather — 因为 Linear 的权重拆分方式和输入分片方式不匹配

### Step 3: GELU Activation（序列维度分片）

In [ ]:
# Rank 0
A0 = GELU(Z0)   # [2, 4096, 8192]

# Rank 1
A1 = GELU(Z1)   # [2, 4096, 8192]

✅ 不需要通信 — element-wise


### Step 4: 第二个 Linear（Row Parallel）
W2 shape: [16384, 4096]，按行拆分：
Rank 0: W_(2,0) shape: [8192, 4096]
Rank 1: W_(2,1) shape: [8192, 4096]

In [ ]:
# Rank 0
B0_partial = A0[:, :, 0:8192] @ W_{2,0}   # [2, 4096, 4096]

# Rank 1
B1_partial = A1[:, :, 8192:16384] @ W_{2,1}   # [2, 4096, 4096]

# All-Reduce
B = all_reduce(B0_partial + B1_partial)   # [2, 4096, 4096] 每个 rank 都有完整结果

🔴 需要 All-Reduce — Row Parallel Linear 的标准做法

### Step 5: Dropout（序列维度分片）

In [ ]:
# 先将 B 按序列维度分片
B0 = B[:, 0:2048, :]    # [2, 2048, 4096]
B1 = B[:, 2048:4096, :] # [2, 2048, 4096]

# Rank 0
C0 = Dropout(B0)

# Rank 1
C1 = Dropout(B1)

✅ 不需要通信

## 总结：通信模式对比

| 操作                         | 输入分片方式 | 输出分片方式 | 是否需要通信 | 通信类型            |
| -------------------------- | ------ | ------ | ------ | --------------- |
| **LayerNorm**              | 序列维度   | 序列维度   | ❌ 不需要  | —               |
| **Dropout**                | 序列维度   | 序列维度   | ❌ 不需要  | —               |
| **GELU/ReLU**              | 序列维度   | 序列维度   | ❌ 不需要  | —               |
| **Column Parallel Linear** | 需要完整序列 | 特征维度分片 | ✅ 需要   | All-Gather 输入   |
| **Row Parallel Linear**    | 特征维度分片 | 需要完整序列 | ✅ 需要   | All-Reduce 输出   |
| **Attention (Softmax)**    | 序列维度   | 序列维度   | ✅ 需要   | All-Gather/Ring |

## 为什么这个优化很重要？

### 通信量计算
假设：
- batch_size = b
- seq_len = s
- hidden_dim = h
- TP size = t

### 标准 TP（输入复制到所有 ranks）：
- 每个 rank 的激活内存：b×s×h 
- 总激活内存（跨所有 ranks）：t×b×s×h 
### Sequence Parallelism（输入按序列分片）：
- 每个 rank 的激活内存：b×(s/t)×h 
- 总激活内存（跨所有 ranks）：b×s×h （和单卡一样！）
节省：激活内存减少了 t  倍！

### 实际效果
在 GPT-3 175B 模型中：
- 标准 TP：LayerNorm + Dropout 的激活需要完整复制
- 使用 SP 后：这些层的激活可以分片，节省大量显存
- 对于长序列（如 32K, 100K+），这个优化至关重要

## Megatron-LM 中的实现
在 Megatron-LM 的 transformer.py 中，Sequence Parallelism 的实现：


In [ ]:
class ParallelTransformerLayer(MegatronModule):
    def __init__(self, config):
        # ...
        self.sequence_parallel = config.sequence_parallel
        
        # LayerNorm 可以在序列维度分片
        self.input_layernorm = LayerNorm(...)
        
        # Attention 需要特殊处理
        self.self_attention = ParallelAttention(config)
        
        # FFN
        self.mlp = ParallelMLP(config)
    
    def forward(self, hidden_states):
        # hidden_states: 如果在 SP 模式下，是序列维度分片的
        
        # 1. LayerNorm: 不需要通信
        layernorm_output = self.input_layernorm(hidden_states)
        
        # 2. Attention: 需要 All-Gather 序列（在 Attention 内部处理）
        attention_output = self.self_attention(layernorm_output)
        
        # 3. Dropout: 不需要通信
        hidden_states = hidden_states + self.dropout(attention_output)
        
        # 4. 第二个 LayerNorm: 不需要通信
        layernorm_output = self.post_attention_layernorm(hidden_states)
        
        # 5. MLP: Linear 需要 All-Gather，GELU/Dropout 不需要
        mlp_output = self.mlp(layernorm_output)
        
        # 6. 残差 + Dropout
        output = hidden_states + self.dropout(mlp_output)
        
        return output

## 关键洞察
### "可以在序列维度分片" 的本质原因：
<font color='red'>LayerNorm、Dropout、Activation 都是 "局部操作"（每个输出元素只依赖于对应的输入元素，不依赖其他 token），</font>因此：
- 输入在序列维度分片 → 输出也在序列维度分片
- 不需要知道其他 rank 上的 token

而 Linear 层是 "全局操作"（每个输出特征是所有输入特征的线性组合），且权重矩阵按特征维度拆分，因此：
- 输入需要在特征维度完整（或按另一种方式分片）
- 当输入在序列维度分片时，必须 All-Gather 才能进行矩阵乘法

这个区分使得 Sequence Parallelism 成为一种有效的显存优化策略：在不需要全局交互的地方（LayerNorm/Dropout）分片，在需要的地方（Linear/Attention）聚合。